# 15 - Reranking Answer-Quality Test: Real End-to-End Synthesis, 3 Configs x 10 P3.v3 Gold Questions

**The gap this notebook fills:** `RERANKING_IMPACT_ANALYSIS.md` and
`guidance/ANALYSIS_reranker_judgment_calls_2026-07-29.md` measured reranking's effect
only at the retrieval-stage proxy level (recall@k, MRR on sentence-ID lists) -- "flat
but safe." Nobody has yet looked at whether reranking changes the actual **final answer**
the LLM produces. That is what this notebook tests.

## Design

Real `answer_query()`-equivalent synthesis (real Bedrock Claude Haiku 4.5 calls, real
cost, ~$0.017-0.02/query) on the 10 `gold_version == "P3.v3"` questions from
`MLFlow_POC/data/p3_gold_test_suite_31q.json`, at **three configs**:

| Config | `enable_reranking` | `rerank_top_n_blocks` | What it is |
| :-- | :-- | :-- | :-- |
| **A** | `false` | n/a | current shipped default (reranking off) |
| **B** | `true` | `16` | reranking on, looser prune |
| **C** | `true` | `8` | reranking on, shipped default prune when on |

30 real synthesis calls total, budgeted at ~$0.50-0.60.

## Why not just call `answer_query()` three times with different config?

`answer_query()` (`synthesis_pipeline/orchestrator.py`) always calls
`init_rag_components()` (`supply_lines.py`) with **zero arguments**, which always
constructs its own fresh `MLConfig()` internally -- there is no parameter anywhere in
that call chain to inject a different retrieval config. And editing
`.aws_config/ml_config.yaml` on disk is unsafe here: another process in this same repo
is concurrently loading `MLConfig()` for unrelated work, and a mid-edit read of that
file would silently corrupt its config.

So this notebook verifies (empirically, next cell) whether `MLConfig()` is a singleton
within one Python process, then uses whichever override strategy that verification
licenses. Spoiler, confirmed below: **`MLConfig()` is not a singleton** -- every call
re-reads and re-parses the YAML from disk into a brand new object. That rules out
"mutate a config object I'm holding and hope `init_rag_components()` picks it up" (it
constructs its own). The safe path is instead: build **my own** `MLConfig()` instance,
mutate **that instance's** in-memory `.cfg['retrieval']` dict (never written to disk,
never shared with any other process), and pass **that instance** through a hand-built
mirror of `init_rag_components()` that accepts a config object instead of constructing
one internally. Everything downstream (`build_combined_context()`, `PromptLoader`,
`BedrockClient.invoke()`, `create_success_response()`) is the **real, unmodified**
production code -- only the config-construction step differs from `answer_query()`.

One deliberate omission: `QueryLogger` (which persists to local parquet + S3) is not
called, to avoid writing into the shared query-log store while other work is running
concurrently in this repo. This has no effect on the metrics this notebook measures
(answer text, token/cost/context metadata) -- it only skips a persistence side effect
this analysis doesn't need.


In [1]:
import sys
import json
import time
import copy
from pathlib import Path
from datetime import datetime

for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == "ModelPipeline":
        MODEL_ROOT = p
        break
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

print("MODEL_ROOT:", MODEL_ROOT)


MODEL_ROOT: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline


In [2]:
from finrag_ml_tg1.loaders.ml_config_loader import MLConfig
from finrag_ml_tg1.loaders.data_loader_factory import create_data_loader

from finrag_ml_tg1.rag_modules_src.entity_adapter.entity_adapter import EntityAdapter
from finrag_ml_tg1.rag_modules_src.metric_pipeline.src.pipeline import MetricPipeline
from finrag_ml_tg1.rag_modules_src.utilities.query_embedder_v2 import (
    QueryEmbedderV2, EmbeddingRuntimeConfig,
)
from finrag_ml_tg1.rag_modules_src.rag_pipeline.metadata_filters import MetadataFilterBuilder
from finrag_ml_tg1.rag_modules_src.rag_pipeline.variant_pipeline import VariantPipeline
from finrag_ml_tg1.rag_modules_src.rag_pipeline.s3_retriever import S3VectorsRetriever
from finrag_ml_tg1.rag_modules_src.rag_pipeline.sentence_expander import SentenceExpander
from finrag_ml_tg1.rag_modules_src.rag_pipeline.context_assembler import ContextAssembler
from finrag_ml_tg1.rag_modules_src.rag_pipeline.reranker import CohereReranker

from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.supply_lines import (
    RAGComponents, build_combined_context,
)
from finrag_ml_tg1.rag_modules_src.prompts.prompt_loader import PromptLoader
from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.bedrock_client import (
    create_bedrock_client_from_config,
)
from finrag_ml_tg1.rag_modules_src.synthesis_pipeline.models import create_success_response

print("Imports OK")


Imports OK


## Step 1 - Verify empirically: is `MLConfig()` a singleton in this process?

Two independent `MLConfig()` calls should have **different** `id()`s if it is not a
singleton, and mutating one instance's `.cfg` dict should have **zero** effect on a
later, independently-constructed instance. Confirmed once already outside this notebook
via a throwaway script; re-run here so the check is reproducible and part of the actual
deliverable, not just asserted in prose.


In [3]:
c1 = MLConfig()
c2 = MLConfig()
print("id(c1):", id(c1), " id(c2):", id(c2), " same object?", c1 is c2)

before = c1.cfg['retrieval']['rerank_top_n_blocks']
c1.cfg['retrieval']['rerank_top_n_blocks'] = 999  # in-memory only, never touches disk
after = c1.cfg['retrieval']['rerank_top_n_blocks']

c3 = MLConfig()  # fresh instance, independent of c1's mutation
c3_val = c3.cfg['retrieval']['rerank_top_n_blocks']

print(f"c1 rerank_top_n_blocks: {before} -> {after} (mutated in place, as expected)")
print(f"c3 (fresh MLConfig()) rerank_top_n_blocks: {c3_val} (must be the yaml default, 8)")

assert c1 is not c2, "MLConfig() unexpectedly returned the same object -- singleton assumption would be WRONG"
assert c3_val == 8, "Fresh MLConfig() picked up the mutation from c1 -- something IS shared across instances, investigate before proceeding"

print()
print("VERIFIED: MLConfig() is NOT a singleton. Mutating one instance's .cfg dict")
print("is invisible to any other MLConfig() instance and never touches the yaml file.")
print("Therefore: build a dedicated config instance per arm, mutate ONLY its own")
print(".cfg['retrieval'] in memory, and pass THAT instance through a hand-built")
print("component factory -- never rely on init_rag_components()/answer_query()'s")
print("internal fresh MLConfig() to see an external mutation, because it can't.")


[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
id(c1): 13569218064  id(c2): 6301791056  same object? False
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
c1 rerank_top_n_blocks: 8 -> 999 (mutated in place, as expected)
c3 (fresh MLConfig()) rerank_top_n_blocks: 8 (must be the yaml default, 8)

VERIFIED: MLConfig() is NOT a singleton. Mutating one instance's .cfg dict
is invisible to any other MLConfig() instance and never touches the yaml file.
Therefore: build a dedicated config instance per arm, mutate ONLY its own
.cfg

## Step 2 - Config-injectable mirror of `init_rag_components()` / `answer_query()`

`init_rag_components_with_overrides()` below is a line-for-line mirror of
`supply_lines.init_rag_components()`, with one difference: it takes a `retrieval_overrides`
dict and applies it to its **own** freshly-constructed `MLConfig()` instance before
building any component, so `S3VectorsRetriever`/`CohereReranker` see the overridden
values. Every downstream call (`build_combined_context()`, `PromptLoader`,
`BedrockClient.invoke()`, `create_success_response()`) is the unmodified production
function.


In [4]:
def init_rag_components_with_overrides(retrieval_overrides: dict):
    """Mirrors supply_lines.init_rag_components(), but builds its own MLConfig()
    and applies retrieval_overrides to that instance's in-memory .cfg['retrieval']
    before constructing anything. Never touches ml_config.yaml on disk."""
    config = MLConfig()
    config.cfg['retrieval'].update(retrieval_overrides)

    bedrock_client = config.get_bedrock_client()
    data_loader = create_data_loader(config)

    adapter = EntityAdapter(data_loader=data_loader)
    metric_pipeline = MetricPipeline(data_loader=data_loader)

    embedding_cfg = config.cfg["embedding"]
    runtime_cfg = EmbeddingRuntimeConfig.from_ml_config(embedding_cfg)
    embedder = QueryEmbedderV2(runtime_cfg, boto_client=bedrock_client)

    filter_builder = MetadataFilterBuilder(config)
    variant_pipeline = VariantPipeline(config, adapter, embedder, bedrock_client)

    retrieval_cfg = config.get_retrieval_config()
    retriever = S3VectorsRetriever(
        retrieval_config=retrieval_cfg,
        aws_access_key_id=config.aws_access_key,
        aws_secret_access_key=config.aws_secret_key,
        region=config.region,
        variant_pipeline=variant_pipeline,
    )

    expander = SentenceExpander(data_loader=data_loader, config=config)
    assembler = ContextAssembler(data_loader=data_loader, config=config)

    reranker = None
    if retrieval_cfg.get("enable_reranking"):
        reranker = CohereReranker(
            retrieval_config=retrieval_cfg,
            region=config.region,
            aws_access_key_id=config.aws_access_key,
            aws_secret_access_key=config.aws_secret_key,
        )

    rag = RAGComponents(
        adapter=adapter, metric_pipeline=metric_pipeline, embedder=embedder,
        filter_builder=filter_builder, variant_pipeline=variant_pipeline,
        retriever=retriever, expander=expander, assembler=assembler, reranker=reranker,
    )
    return rag, config


def answer_query_with_overrides(query: str, retrieval_overrides: dict, model_key=None) -> dict:
    """Mirrors orchestrator.answer_query(), minus QueryLogger (deliberately skipped,
    see markdown above), with a config-override injection point that answer_query()
    itself does not expose."""
    start_time = time.time()

    rag, config = init_rag_components_with_overrides(retrieval_overrides)
    prompt_loader = PromptLoader()
    llm_client = create_bedrock_client_from_config(config, model_key)

    combined_context, context_metadata = build_combined_context(
        query=query, rag=rag, include_kpi=True, include_rag=True,
    )

    system_prompt = prompt_loader.load_system_prompt()
    user_prompt = prompt_loader.format_query_template(combined_context)

    llm_response = llm_client.invoke(system=system_prompt, user=user_prompt)

    processing_time_ms = (time.time() - start_time) * 1000
    response = create_success_response(
        query=query,
        answer=llm_response['content'],
        context=combined_context,
        llm_response=llm_response,
        context_metadata=context_metadata,
        processing_time_ms=processing_time_ms,
    )
    return response.to_dict()

print("answer_query_with_overrides() defined")


answer_query_with_overrides() defined


## Step 3 - Sanity-check the override actually reaches the reranker (no AWS calls)

Cheap structural check before spending any money: build components under each of the
three configs and confirm `rag.reranker` is `None`/present with the right `top_n`, exactly
as the shipped `init_rag_components()` conditional would produce.


In [5]:
CONFIGS = {
    "A_no_rerank":     {"enable_reranking": False},
    "B_rerank_top16":  {"enable_reranking": True, "rerank_top_n_blocks": 16},
    "C_rerank_top8":   {"enable_reranking": True, "rerank_top_n_blocks": 8},
}

for label, overrides in CONFIGS.items():
    rag, config = init_rag_components_with_overrides(overrides)
    rc = config.get_retrieval_config()
    reranker_state = "None" if rag.reranker is None else f"top_n={rag.reranker.top_n}"
    print(f"{label:18s} enable_reranking={rc.get('enable_reranking')!s:6s} "
          f"rerank_top_n_blocks={rc.get('rerank_top_n_blocks')!s:5s} rag.reranker={reranker_state}")

print()
print("Confirmed: each config produces the expected rag.reranker state.")


[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
A_no_rerank        enable_reranking=False  rerank_top_n_blocks=8     rag.reranker=None
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env


[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode


B_rerank_top16     enable_reranking=True   rerank_top_n_blocks=16    rag.reranker=top_n=16
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
C_rerank_top8      enable_reranking=True   rerank_top_n_blocks=8     rag.reranker=top_n=8

Confirmed: each config produces the expected rag.reranker state.


## Step 4 - Load the 10 P3.v3 gold questions


In [6]:
GOLD_PATH = MODEL_ROOT.parent / "MLFlow_POC" / "data" / "p3_gold_test_suite_31q.json"
assert GOLD_PATH.exists(), f"Gold file not found: {GOLD_PATH}"

with open(GOLD_PATH, encoding="utf-8") as f:
    all_gold = json.load(f)

gold_v3 = [g for g in all_gold if g.get("gold_version") == "P3.v3"]
print(f"Loaded {len(all_gold)} total gold questions, {len(gold_v3)} at gold_version == 'P3.v3'")
assert len(gold_v3) == 10, f"Expected 10 P3.v3 questions, found {len(gold_v3)}"

for g in gold_v3:
    print(f"  {g['question_id']:12s} {g['retrieval_scope']:14s} {g['answer_type']:8s} {g['difficulty']:8s} {g['question_text'][:70]}...")


Loaded 31 total gold questions, 10 at gold_version == 'P3.v3'
  P3V3-Q001    cross_year     span     medium   Across its fiscal 2018-2020 10-K filings, how does Walmart Inc. explai...
  P3V3-Q002    cross_year     span     hard     Over time, how does Meta Platforms describe the regulatory and policy ...
  P3V3-Q003    cross_year     span     medium   Between 2022 and 2024, how does Johnson & Johnson describe the impact ...
  P3V3-Q004    cross_company  list     medium   In their 2009 Form 10-K risk-factor disclosures, how do Radian Group, ...
  P3V3-Q005    cross_company  list     hard     For 2010, how do Walmart, Apple, Microsoft and Icahn Enterprises descr...
  P3V3-Q006    cross_company  list     medium   In their 2023 Form 10-K filings, what revenue-related themes do Exxon ...
  P3V3-Q007    local          span     easy     Where does Tesla define Adjusted EBITDA in its 2022 Form 10-K, and how...
  P3V3-Q008    local          span     easy     How does Icahn Enterprises define Ad

## Step 5 - Run: 30 real synthesis calls (10 questions x 3 configs)

**This cell spends real money** (~$0.017-0.02/call x 30 = ~$0.50-0.60). Each call is
wrapped in `try/except` so one failure doesn't lose the rest of the run; results are
appended incrementally and checkpointed to disk after every call.


In [7]:
RESULTS_PATH = Path.cwd() / "15_reranking_answer_quality_results_30q.json"

results = []
t_run_start = time.time()

for qi, gq in enumerate(gold_v3, start=1):
    for ci, (config_label, overrides) in enumerate(CONFIGS.items(), start=1):
        tag = f"[{qi:2d}/{len(gold_v3)} x {ci}/{len(CONFIGS)}] {gq['question_id']} / {config_label}"
        t0 = time.time()
        try:
            result = answer_query_with_overrides(gq["question_text"], overrides)
            elapsed = time.time() - t0
            row = {
                "question_id": gq["question_id"],
                "config_label": config_label,
                "retrieval_overrides": overrides,
                "query": gq["question_text"],
                "gold_answer_text": gq["answer_text"],
                "answer_type": gq["answer_type"],
                "retrieval_scope": gq["retrieval_scope"],
                "difficulty": gq["difficulty"],
                "answer": result["answer"],
                "context_length": result["metadata"]["context"]["context_length"],
                "retrieval_stats": result["metadata"]["context"].get("retrieval_stats"),
                "llm_input_tokens": result["metadata"]["llm"]["input_tokens"],
                "llm_output_tokens": result["metadata"]["llm"]["output_tokens"],
                "llm_total_tokens": result["metadata"]["llm"]["total_tokens"],
                "llm_cost": result["metadata"]["llm"]["cost"],
                "stop_reason": result["metadata"]["llm"]["stop_reason"],
                "processing_time_ms": result["metadata"]["processing_time_ms"],
                "wall_clock_s": round(elapsed, 1),
                "error": None,
            }
            print(f"{tag}  OK   ${row['llm_cost']:.4f}  in={row['llm_input_tokens']} out={row['llm_output_tokens']}  ctx_len={row['context_length']}  {elapsed:.1f}s")
        except Exception as e:
            row = {
                "question_id": gq["question_id"],
                "config_label": config_label,
                "retrieval_overrides": overrides,
                "query": gq["question_text"],
                "gold_answer_text": gq["answer_text"],
                "answer_type": gq["answer_type"],
                "retrieval_scope": gq["retrieval_scope"],
                "difficulty": gq["difficulty"],
                "answer": None,
                "error": f"{type(e).__name__}: {e}",
            }
            print(f"{tag}  FAILED: {type(e).__name__}: {e}")

        results.append(row)

        # Checkpoint after every call -- real money already spent, don't risk losing it.
        with open(RESULTS_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        time.sleep(1.5)  # small courtesy delay; other work may be hitting Bedrock concurrently

total_elapsed_min = (time.time() - t_run_start) / 60
print()
print(f"Run complete: {len(results)} rows in {total_elapsed_min:.1f} min. Saved to {RESULTS_PATH}")


[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode


[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env


[DEBUG] Raw length: 7402
[DEBUG] Cleaned length: 7380
[DEBUG] First 200 chars: Walmart's approach to long-term debt management during fiscal 2018-2020 was driven by two distinct strategic priorities that shifted significantly across this period. The company's debt decisions were
[ 1/10 x 1/3] P3V3-Q001 / A_no_rerank  OK   $0.0197  in=10333 out=1868  ctx_len=24231  21.1s


[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env


[DEBUG] Raw length: 7002
[DEBUG] Cleaned length: 6984
[DEBUG] First 200 chars: Walmart's approach to long-term debt management during fiscal 2018-2020 was shaped by two major strategic initiatives: debt optimization in fiscal 2018 and the transformative Flipkart acquisition in f
[ 1/10 x 2/3] P3V3-Q001 / B_rerank_top16  OK   $0.0192  in=10518 out=1730  ctx_len=24884  19.4s


...

[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env


[DEBUG] Raw length: 3186
[DEBUG] Cleaned length: 3168
[DEBUG] First 200 chars: Meta explicitly attributes movements in "Other income/(expense), net" to foreign currency remeasurement in both its 2015 and 2016 Form 10-K filings, with detailed quantification of the foreign exchang
[10/10 x 2/3] P3V3-Q010 / B_rerank_top16  OK   $0.0149  in=10784 out=826  ctx_len=24216  11.6s


[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env


[DEBUG] Raw length: 4689
[DEBUG] Cleaned length: 4629
[DEBUG] First 200 chars: Meta explicitly attributes movements in "Other income/(expense), net" to foreign currency remeasurement in both its 2015 and 2016 Form 10-K filings, with detailed explanations of the underlying dynami
[10/10 x 3/3] P3V3-Q010 / C_rerank_top8  OK   $0.0147  in=8539 out=1241  ctx_len=14441  14.1s



Run complete: 30 rows in 9.0 min. Saved to /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline/finrag_ml_tg1/validation_notebooks/15_reranking_answer_quality_results_30q.json


In [8]:
ok_rows = [r for r in results if r.get("error") is None]
failed_rows = [r for r in results if r.get("error") is not None]
total_cost = sum(r["llm_cost"] for r in ok_rows)

print(f"Succeeded: {len(ok_rows)}/{len(results)}")
print(f"Failed:    {len(failed_rows)}/{len(results)}")
for r in failed_rows:
    print(f"  FAILED {r['question_id']} / {r['config_label']}: {r['error']}")
print(f"Total real LLM cost this run: ${total_cost:.4f}")


Succeeded: 30/30
Failed:    0/30
Total real LLM cost this run: $0.4920


## Step 6 - Score each answer against its gold reference

`rag_modules_src/utilities/evaluation_metrics.py` defines `evaluate_answer()` with
ROUGE-L, BERTScore, cosine similarity and BLEURT. Its actual signature/behavior was read
directly from source before writing anything below (not guessed):

```python
def evaluate_answer(gold_answer, synthesis_answer, include_bleurt=True, include_timing=True) -> dict
```

It requires the `bert_score`, `rouge_score`, `sentence_transformers` and (optionally)
`bleurt` packages. **Checked empirically across every environment on this machine**
(`finsights_revival`, `mjs_mlcvdl_unified_m5`, `finsight-venv`, `base`): `bert_score`,
`rouge_score`, and `bleurt` are installed in **none** of them, and no `BLEURT-20`
checkpoint exists anywhere on disk (it would need a ~2GB download on first use, on top
of the missing package). They *are* listed in `environments/requirements.txt` as
intended dev-only dependencies, so this is a genuine environment gap, not an invented
requirement -- but installing new packages is outside this session's authority (hard
guardrail: no new software installs without explicit sign-off), so `evaluate_answer()`
itself cannot be called here.

**What this notebook computes instead, with zero new installs:**
- **Cosine similarity** -- via `sentence_transformers` (already installed), using the
  *same* model constant `evaluation_metrics.py` uses (`all-MiniLM-L6-v2`) and the same
  `util.cos_sim` call. This is not an approximation; it is the identical computation
  `evaluate_answer()` would have run for this one metric.
- **ROUGE-L** -- reimplemented here via the standard LCS-based F-measure (the same
  algorithm `rouge_score.rouge_scorer` implements for `'rougeL'`), word-tokenized,
  lowercased, **without** Porter stemming (the packaged scorer runs with
  `use_stemmer=True`). Flagged as an approximation for that reason -- scores will run
  slightly lower than the packaged metric would report, since unstemmed exact-token
  matching is strictly harder to satisfy.
- **BERTScore F1**: **not computed** (package not installed).
- **BLEURT**: **not computed** (package not installed, no checkpoint present, and it is
  independently the slowest of the four per `RETRIEVAL_IMPROVEMENT_STUDY.md`
  ~7.2s/pair -- this notebook has two independent reasons to skip it, not one).

This gap is reported plainly in `RERANKING_ANSWER_QUALITY_TEST.md` rather than
papered over.


In [9]:
import re
from sentence_transformers import SentenceTransformer, util

SENTENCE_MODEL = SentenceTransformer('all-MiniLM-L6-v2')  # same constant evaluation_metrics.py uses


def _tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())


def rouge_l_fmeasure(reference: str, candidate: str) -> float:
    """LCS-based ROUGE-L F-measure, word-tokenized, no stemming.
    Same algorithm rouge_score.rouge_scorer uses for 'rougeL'; see markdown above
    for the one documented difference (no Porter stemming here)."""
    ref = _tokenize(reference)
    cand = _tokenize(candidate)
    if not ref or not cand:
        return 0.0

    # Standard LCS length via DP.
    n, m = len(ref), len(cand)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref[i - 1] == cand[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[n][m]

    if lcs == 0:
        return 0.0
    recall = lcs / n
    precision = lcs / m
    beta = 1.0  # F1 (rouge_scorer's default 'fmeasure' is beta=1)
    return (1 + beta ** 2) * precision * recall / (recall + beta ** 2 * precision)


def cosine_similarity(reference: str, candidate: str) -> float:
    emb_ref = SENTENCE_MODEL.encode(reference, convert_to_tensor=False)
    emb_cand = SENTENCE_MODEL.encode(candidate, convert_to_tensor=False)
    return float(util.cos_sim(emb_ref, emb_cand).item())


print("Scoring functions defined")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Scoring functions defined


In [10]:
def gold_reference_text(gold_answer):
    if isinstance(gold_answer, list):
        return "\n\n".join(gold_answer)
    return gold_answer


for row in results:
    if row.get("answer") is None:
        row["rouge_l"] = None
        row["cosine_sim"] = None
        continue
    ref = gold_reference_text(row["gold_answer_text"])
    row["rouge_l"] = round(rouge_l_fmeasure(ref, row["answer"]), 3)
    row["cosine_sim"] = round(cosine_similarity(ref, row["answer"]), 3)

for row in results[:3]:
    print(row["question_id"], row["config_label"], "rouge_l=", row.get("rouge_l"), "cosine_sim=", row.get("cosine_sim"))


P3V3-Q001 A_no_rerank rouge_l= 0.07 cosine_sim= 0.843
P3V3-Q001 B_rerank_top16 rouge_l= 0.073 cosine_sim= 0.825
P3V3-Q001 C_rerank_top8 rouge_l= 0.078 cosine_sim= 0.838


## Step 7 - Comparison table: per question x config


In [11]:
header = f"{'question_id':12s} {'config':16s} {'rouge_l':>8s} {'cosine':>8s} {'ctx_len':>8s} {'in_tok':>7s} {'out_tok':>7s} {'cost':>8s}"
print(header)
print("-" * len(header))
for row in results:
    if row.get("answer") is None:
        print(f"{row['question_id']:12s} {row['config_label']:16s}   FAILED: {row['error']}")
        continue
    print(f"{row['question_id']:12s} {row['config_label']:16s} "
          f"{row['rouge_l']:8.3f} {row['cosine_sim']:8.3f} {row['context_length']:8d} "
          f"{row['llm_input_tokens']:7d} {row['llm_output_tokens']:7d} ${row['llm_cost']:7.4f}")


question_id  config            rouge_l   cosine  ctx_len  in_tok out_tok     cost
---------------------------------------------------------------------------------
P3V3-Q001    A_no_rerank         0.070    0.843    24231   10333    1868 $ 0.0197
P3V3-Q001    B_rerank_top16      0.073    0.825    24884   10518    1730 $ 0.0192
P3V3-Q001    C_rerank_top8       0.078    0.838    14336    8081    1572 $ 0.0159
P3V3-Q002    A_no_rerank         0.049    0.732    60281   16340    3466 $ 0.0337
P3V3-Q002    B_rerank_top16      0.055    0.639    36535   11747    2447 $ 0.0240
P3V3-Q002    C_rerank_top8       0.051    0.763    23503    9088    2862 $ 0.0234
P3V3-Q003    A_no_rerank         0.087    0.850    46475   17768    1414 $ 0.0248
P3V3-Q003    B_rerank_top16      0.078    0.805    33664   14548    1741 $ 0.0233
P3V3-Q003    C_rerank_top8       0.118    0.792    16021    8814     958 $ 0.0136
P3V3-Q004    A_no_rerank         0.079    0.788    25450    9840    1320 $ 0.0164
P3V3-Q004    B_r

In [12]:
from collections import defaultdict

by_config = defaultdict(list)
for row in results:
    if row.get("answer") is not None:
        by_config[row["config_label"]].append(row)

print(f"{'config':16s} {'n':>3s} {'avg_rouge_l':>12s} {'avg_cosine':>11s} {'avg_ctx_len':>12s} {'avg_in_tok':>11s} {'avg_out_tok':>12s} {'total_cost':>11s}")
agg = {}
for label, rows in by_config.items():
    n = len(rows)
    avg_rouge = sum(r["rouge_l"] for r in rows) / n
    avg_cos = sum(r["cosine_sim"] for r in rows) / n
    avg_ctx = sum(r["context_length"] for r in rows) / n
    avg_in = sum(r["llm_input_tokens"] for r in rows) / n
    avg_out = sum(r["llm_output_tokens"] for r in rows) / n
    total_cost = sum(r["llm_cost"] for r in rows)
    agg[label] = dict(n=n, avg_rouge_l=avg_rouge, avg_cosine=avg_cos, avg_context_length=avg_ctx,
                       avg_input_tokens=avg_in, avg_output_tokens=avg_out, total_cost=total_cost)
    print(f"{label:16s} {n:3d} {avg_rouge:12.3f} {avg_cos:11.3f} {avg_ctx:12.0f} {avg_in:11.0f} {avg_out:12.0f} ${total_cost:10.4f}")


config             n  avg_rouge_l  avg_cosine  avg_ctx_len  avg_in_tok  avg_out_tok  total_cost
A_no_rerank       10        0.101       0.762        35905       13108         1318 $    0.1970
B_rerank_top16    10        0.105       0.732        25015       10689         1124 $    0.1631
C_rerank_top8     10        0.112       0.770        14314        8140         1010 $    0.1319


## Step 8 - Per-question deltas across configs (A vs B, A vs C, B vs C)

Looks for any question where the three configs produced meaningfully different
cosine-similarity-to-gold or context length -- candidates for the "concrete example"
excerpts in the write-up.


In [13]:
by_question = defaultdict(dict)
for row in results:
    by_question[row["question_id"]][row["config_label"]] = row

print(f"{'question_id':12s} {'cos_A':>7s} {'cos_B':>7s} {'cos_C':>7s} {'spread':>7s} {'ctxA':>6s} {'ctxB':>6s} {'ctxC':>6s}")
spreads = []
for qid, cfgs in by_question.items():
    a = cfgs.get("A_no_rerank")
    b = cfgs.get("B_rerank_top16")
    c = cfgs.get("C_rerank_top8")
    if not (a and b and c) or a.get("cosine_sim") is None or b.get("cosine_sim") is None or c.get("cosine_sim") is None:
        print(f"{qid:12s}  incomplete row(s), skipping")
        continue
    cos_vals = [a["cosine_sim"], b["cosine_sim"], c["cosine_sim"]]
    spread = max(cos_vals) - min(cos_vals)
    spreads.append((spread, qid))
    print(f"{qid:12s} {a['cosine_sim']:7.3f} {b['cosine_sim']:7.3f} {c['cosine_sim']:7.3f} {spread:7.3f} "
          f"{a['context_length']:6d} {b['context_length']:6d} {c['context_length']:6d}")

spreads.sort(reverse=True)
print()
print("Questions ranked by cosine-similarity spread across configs (largest first):")
for spread, qid in spreads:
    print(f"  {qid}: spread={spread:.3f}")


question_id    cos_A   cos_B   cos_C  spread   ctxA   ctxB   ctxC
P3V3-Q001      0.843   0.825   0.838   0.018  24231  24884  14336
P3V3-Q002      0.732   0.639   0.763   0.124  60281  36535  23503
P3V3-Q003      0.850   0.805   0.792   0.058  46475  33664  16021
P3V3-Q004      0.788   0.637   0.727   0.151  25450  17297   8627
P3V3-Q005      0.639   0.636   0.736   0.100  24678   8063   4110
P3V3-Q006      0.766   0.784   0.770   0.018  42604  28093  16808
P3V3-Q007      0.764   0.731   0.735   0.033  46313  38974  27112
P3V3-Q008      0.757   0.772   0.787   0.030  22840   9313   5380
P3V3-Q009      0.814   0.810   0.854   0.044  29107  29107  12801
P3V3-Q010      0.665   0.685   0.703   0.038  37074  24216  14441

Questions ranked by cosine-similarity spread across configs (largest first):
  P3V3-Q004: spread=0.151
  P3V3-Q002: spread=0.124
  P3V3-Q005: spread=0.100
  P3V3-Q003: spread=0.058
  P3V3-Q009: spread=0.044
  P3V3-Q010: spread=0.038
  P3V3-Q007: spread=0.033
  P3V3-Q008: s

## Step 9 - Save final scored results for the write-up


In [14]:
FINAL_PATH = Path.cwd() / "15_reranking_answer_quality_scored_30q.json"
with open(FINAL_PATH, "w", encoding="utf-8") as f:
    json.dump({"results": results, "aggregate_by_config": agg}, f, indent=2, ensure_ascii=False)

print(f"Saved scored results to {FINAL_PATH}")
print(f"Raw (pre-scoring) checkpoint remains at {RESULTS_PATH}")


Saved scored results to /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline/finrag_ml_tg1/validation_notebooks/15_reranking_answer_quality_scored_30q.json
Raw (pre-scoring) checkpoint remains at /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline/finrag_ml_tg1/validation_notebooks/15_reranking_answer_quality_results_30q.json


## Notes for the write-up (`RERANKING_ANSWER_QUALITY_TEST.md`)

- This notebook's honest job is to report whether reranking changed the **answer**, not
  to re-litigate the retrieval-proxy-metric result already recorded in
  `RERANKING_IMPACT_ANALYSIS.md`. "No meaningful difference detected" is a legitimate,
  useful finding here, not a failure to find something -- say so plainly if that's what
  the numbers show.
- n=10 per config (30 total) is even smaller than the n=31 retrieval-stage ablation
  already flagged as underpowered in `guidance/ANALYSIS_reranker_judgment_calls_2026-07-29.md`
  Sec 2.2 -- treat any numeric difference here as suggestive, not conclusive, and say so.
- BERTScore/BLEURT gap (Step 6) should be stated plainly in the write-up, not hidden in
  a footnote -- it is a real limitation of this specific run, not a design choice made
  for quality reasons.
